# Day 6 — Baseline Test

This notebook integrates all the core python skills from day-01.

## Validation goals
- Create a structured standalone Python project (`dataset-processor/`)
- CRead and parse CSV input files using `pathlib` and `csv.DictReader`
- Clean and type-cast raw data safely with exception handling
- Calculate 5 core statistics (count, sum, mean, min, max)
- Encapsulate data and operations into a domain class (`DatasetAnalyzer`)
- Define and raise a custom domain exception (`DatasetFileNotFoundError`)
- Write a robust unit test suite using `pytest` and `tmp_path`


### Step 1: 
Creating a standalone Python project with a clear separation of source code, tests, and data.

dataset-processor/
├── data/
│   └── input.csv
├── src/
│   └── processor.py
├── tests/
│   └── test_processor.py
├── requirements.txt
├── README.md
└── .gitignore

### Commands

`mkdir dataset-processor`
`cd dataset-processor`

`python -m venv .venv`
`.venv\Scripts\activate`

`python -m pip install --upgrade pip`
`pip freeze > requirements.txt`

`git init`

Load and parse data/input.csv inside your script, using Python’s built-in csv module to read and parse data/input.csv without requiring external dependencies

In [ ]:
import csv
from pathlib import Path

file_path = Path(__file__).resolve().parent.parent / "data" / "input.csv"

with file_path.open(mode="r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    data = list(reader)

print(data)

Iterate through the data, strip whitespace, check for missing values, and use try/except to handle invalid numeric values without crashing the program

In [ ]:
cleaned_data = []

for row in data:
    try:
        value = row.get("amount", "").strip()

        if not value:
            continue  # Skip missing values

        row["amount"] = float(value)
        cleaned_data.append(row)

    except (ValueError, AttributeError):
        continue  # Skip invalid rows

print(cleaned_data)

### Statistics: 
Calculate the count, sum, mean, minimum, and maximum using Python’s built-in functions.

In [ ]:
values = [row["amount"] for row in cleaned_data]

if values:
    statistics = {
        "count": len(values),
        "sum": sum(values),
        "mean": sum(values) / len(values),
        "min": min(values),
        "max": max(values),
    }

    print(statistics)
else:
    print("No valid numeric data available.")

Create a DatasetAnalyzer class that encapsulates CSV loading, data cleaning, and statistical calculations while maintaining the dataset as instance state.

In [ ]:
import csv
from pathlib import Path


class DatasetAnalyzer:
    def __init__(self, file_path, numeric_column):
        self.file_path = Path(file_path)
        self.numeric_column = numeric_column
        self.data = []
        self.cleaned_data = []

    def load_data(self):
        with self.file_path.open("r", encoding="utf-8", newline="") as file:
            self.data = list(csv.DictReader(file))
        return self.data

    def clean_data(self):
        self.cleaned_data = []

        for row in self.data:
            try:
                value = row.get(self.numeric_column, "").strip()
                if not value:
                    continue

                row[self.numeric_column] = float(value)
                self.cleaned_data.append(row)

            except (ValueError, AttributeError):
                continue

        return self.cleaned_data

    def calculate_statistics(self):
        values = [row[self.numeric_column] for row in self.cleaned_data]

        if not values:
            return {}

        return {
            "count": len(values),
            "sum": sum(values),
            "mean": sum(values) / len(values),
            "min": min(values),
            "max": max(values),
        }

Catch `FileNotFoundError` and raise a clear custom exception so callers can handle the error explicitly.

In [ ]:
class DatasetFileNotFoundError(Exception):
    """Raised when the dataset file does not exist."""
    pass


def load_data(self):
    try:
        with self.file_path.open("r", encoding="utf-8", newline="") as file:
            self.data = list(csv.DictReader(file))
        return self.data

    except FileNotFoundError as exc:
        raise DatasetFileNotFoundError(
            f"Dataset file not found: {self.file_path}"
        ) from exc

### Designing Automated test
1. Successful CSV loading: Verify that `load_data()` reads an existing CSV file and returns the expected rows.
2. Missing file handling: Verify that `load_data()` raises `DatasetFileNotFoundError` when the specified file does not exist.
3. Data cleaning and statistics: Test that `clean_data()` skips missing or invalid numeric values and that `calculate_statistics()` returns the correct count, sum, mean, minimum, and maximum.

Use `tmp_path` in `pytest` to create temporary CSV files for testing without modifying your actual project data.

### Task:
Structure a test function named `test_load_data_file_not_found` that asserts `DatasetFileNotFoundError` is raised when passing a non-existent path to `DatasetAnalyzer`?

Use `pytest` to run the test

In [ ]:
import pytest
from src.processor import DatasetAnalyzer, DatasetFileNotFoundError


def test_load_data_file_not_found(tmp_path):
    missing_file = tmp_path / "nonexistent.csv"
    analyzer = DatasetAnalyzer(missing_file, "amount")

    with pytest.raises(DatasetFileNotFoundError):
        analyzer.load_data()

Tests for the successful data loading and cleaning workflows.

The tests are saved in `tests/test_processor.py`

Run the test with `pytest-v`

In [ ]:
import csv
from src.processor import DatasetAnalyzer


def test_load_data_successfully(tmp_path):
    # Arrange: create a sample CSV file
    csv_file = tmp_path / "input.csv"
    csv_file.write_text(
        "name,amount\nAlice,100\nBob,200\n",
        encoding="utf-8",
    )

    analyzer = DatasetAnalyzer(csv_file, "amount")

    # Act
    data = analyzer.load_data()

    # Assert
    assert len(data) == 2
    assert data[0] == {"name": "Alice", "amount": "100"}
    assert data[1] == {"name": "Bob", "amount": "200"}


def test_clean_data_successfully(tmp_path):
    # Arrange: create CSV with valid, missing, and invalid values
    csv_file = tmp_path / "input.csv"
    csv_file.write_text(
        "name,amount\n"
        "Alice,100\n"
        "Bob,invalid\n"
        "Charlie,\n"
        "David,250.5\n",
        encoding="utf-8",
    )

    analyzer = DatasetAnalyzer(csv_file, "amount")
    analyzer.load_data()

    # Act
    cleaned_data = analyzer.clean_data()

    # Assert
    assert len(cleaned_data) == 2
    assert cleaned_data[0]["amount"] == 100.0
    assert cleaned_data[1]["amount"] == 250.5
    assert all(
        isinstance(row["amount"], float)
        for row in cleaned_data
    )
